# 06 — Stress testing under shifted realized processes

The fitted forecast distributions stay fixed. We instead generate common pseudo-realizations from a block-bootstrap reference law and perturb the innovations that produce those outcomes. This measures how quickly calibration, proper scores, and portfolio VaR credibility deteriorate under distributional change.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
CACHE = ROOT / 'results/notebook_cache'
CACHE.mkdir(parents=True, exist_ok=True)
from innovcal.api.innovations import fit_innovations
from innovcal.dro.shift_experiments import (
    add_stress_degradation,
    evaluate_shifted_realizations,
    generate_shifted_realizations,
)

In [2]:
split = np.load(CACHE / 'financial_split.npz')
var_data = np.load(CACHE / 'fitted_var.npz')
fitted_var = {
    'beta': var_data['beta'],
    'lags': int(var_data['lags']),
    'include_intercept': True,
}
history = np.vstack([split['train'], split['calibration']])[-fitted_var['lags']:]
truth_model = fit_innovations(
    var_data['residuals'], method='block_bootstrap', block_length=10
)

forecasts = {}
for path in sorted(CACHE.glob('forecast_*.npz')):
    model = path.stem.removeprefix('forecast_')
    forecasts[model] = {'forecast_paths': np.load(path)['forecast_paths'][:, :20]}
print('Fixed forecast distributions:', ', '.join(forecasts))

Fixed forecast distributions: block_bootstrap, bootstrap, cdi_var, diffusion, gaussian, student_t, volatility_bootstrap


In [3]:
tables = []
GRIDS = {
    'variance_inflation': [0.0, 0.10, 0.20, 0.40],
    'outlier_contamination': [0.0, 0.02, 0.05, 0.10],
}
for method, epsilons in GRIDS.items():
    for epsilon in epsilons:
        realized, _ = generate_shifted_realizations(
            fitted_var,
            truth_model,
            history,
            horizon=20,
            n_realizations=100,
            method=method,
            epsilon=epsilon,
            seed=321,
        )
        tables.append(evaluate_shifted_realizations(
            forecasts, realized, method=method, epsilon=epsilon
        ))

stress_results = add_stress_degradation(pd.concat(tables, ignore_index=True))
display(stress_results.sort_values(['shift_method', 'epsilon', 'energy_score']))

,innovation_model,shift_method,epsilon,n_realizations,avg_coverage,avg_width,ece,pit_deviation,crps,energy_score,...,portfolio_var_05,portfolio_breach_rate,realized_portfolio_q05,avg_coverage_change,ece_change,pit_deviation_change,crps_change,energy_score_change,interval_score_change,portfolio_breach_rate_change
29,bootstrap,outlier_contamination,0.00,100,0.897625,0.055825,0.081542,0.035125,0.009934,0.023396,...,-0.061178,0.07,-0.073576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00
28,block_bootstrap,outlier_contamination,0.00,100,0.890250,0.054243,0.081667,0.034825,0.009937,0.023412,...,-0.081072,0.04,-0.073576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00
33,student_t,outlier_contamination,0.00,100,0.916375,0.060047,0.091458,0.038150,0.009973,0.023461,...,-0.066776,0.07,-0.073576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00
31,diffusion,outlier_contamination,0.00,100,0.899500,0.056133,0.088917,0.035875,0.009984,0.023490,...,-0.066163,0.07,-0.073576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00
34,volatility_bootstrap,outlier_contamination,0.00,100,0.908625,0.059564,0.089458,0.036875,0.010004,0.023539,...,-0.063499,0.07,-0.073576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00
30,cdi_var,outlier_contamination,0.00,100,0.911500,0.059506,0.099542,0.039675,0.010084,0.023720,...,-0.078149,0.04,-0.073576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00
32,gaussian,outlier_contamination,0.00,100,0.927250,0.063443,0.112208,0.043575,0.010125,0.023781,...,-0.065280,0.07,-0.073576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.00
36,bootstrap,outlier_contamination,0.02,100,0.886250,0.055825,0.077458,0.034450,0.010706,0.025174,...,-0.061178,0.10,-0.078127,-0.011375,-0.004083,-0.000675,0.000772,0.001778,0.012320,0.03
35,block_bootstrap,outlier_contamination,0.02,100,0.878500,0.054243,0.079708,0.034600,0.010716,0.025205,...,-0.081072,0.05,-0.078127,-0.011750,-0.001958,-0.000225,0.000779,0.001793,0.012735,0.01
40,student_t,outlier_contamination,0.02,100,0.905375,0.060047,0.084958,0.037100,0.010731,0.025211,...,-0.066776,0.09,-0.078127,-0.011000,-0.006500,-0.001050,0.000758,0.001750,0.011752,0.02


In [4]:
stress_results.to_csv(CACHE / 'shifted_realization_stress.csv', index=False)
display(stress_results[[
    'innovation_model', 'shift_method', 'epsilon',
    'avg_coverage_change', 'energy_score_change',
    'portfolio_breach_rate', 'portfolio_breach_rate_change',
]])

,innovation_model,shift_method,epsilon,avg_coverage_change,energy_score_change,portfolio_breach_rate,portfolio_breach_rate_change
0,block_bootstrap,variance_inflation,0.00,0.000000,0.000000,0.04,0.00
1,bootstrap,variance_inflation,0.00,0.000000,0.000000,0.07,0.00
2,cdi_var,variance_inflation,0.00,0.000000,0.000000,0.04,0.00
3,diffusion,variance_inflation,0.00,0.000000,0.000000,0.07,0.00
4,gaussian,variance_inflation,0.00,0.000000,0.000000,0.07,0.00
5,student_t,variance_inflation,0.00,0.000000,0.000000,0.07,0.00
6,volatility_bootstrap,variance_inflation,0.00,0.000000,0.000000,0.07,0.00
7,block_bootstrap,variance_inflation,0.10,-0.025000,0.002396,0.06,0.02
8,bootstrap,variance_inflation,0.10,-0.025875,0.002367,0.07,0.00
9,cdi_var,variance_inflation,0.10,-0.021125,0.002217,0.06,0.02
